# Chapter 15: LangGraph 프로젝트 파이프라인 (Project Pipeline)

LangGraph의 고급 패턴을 조합하여 **완전한 엔드-투-엔드 파이프라인**을 구축합니다.

**Blog Post Generator Pipeline:**
```
[START]
   |
[get_topic_info] -----> LLM generates background info on a topic
   |
[dispatch_writers] --> Send API: split into N sections
   |
[write_section] x N --> Each section written in parallel (Map)
   |
[combine_sections] --> Merge all sections into draft (Reduce)
   |
[human_feedback] --> interrupt() for user review
   |
[finalize_post] --> Apply feedback, generate final version
   |
[END]
```

**Sections:**
- 15.0 Setup & Environment
- 15.1 기본 파이프라인 (get_topic_info → write draft)
- 15.2 병렬 작성 노드 (Send API + Map-Reduce)
- 15.3 Human-in-the-loop (interrupt + Command resume)
- 15.4 완성된 파이프라인 (full pipeline with all patterns)
- 15.5 프로덕션 배포 구조 (langgraph.json, graph.py)
- Final Exercises

---
## 15.0 Setup & Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("OPENAI_BASE_URL:", os.getenv("OPENAI_BASE_URL"))
print("OPENAI_MODEL_NAME:", os.getenv("OPENAI_MODEL_NAME"))

In [ ]:
# 패키지 확인
from importlib.metadata import version

print(f"langgraph: {version('langgraph')}")
print(f"langchain: {version('langchain')}")

---
## 15.1 기본 파이프라인 — 2노드 선형 그래프

가장 단순한 파이프라인: 주제 조사 → 초고 작성

```
START → get_topic_info → write_draft → END
```

핵심 개념:
- `TypedDict` 상태 정의 — 파이프라인 데이터 흐름
- `init_chat_model` — Azure OpenAI LLM 초기화
- 선형 엣지 — 노드 간 순차 연결

In [ ]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# 파이프라인 상태 정의
class PipelineState(TypedDict):
    topic: str
    background_info: str
    draft: str


# 노드 1: 주제 조사
def get_topic_info(state: PipelineState):
    topic = state["topic"]
    response = llm.invoke(
        f"Provide a concise background summary (3-4 paragraphs) about: {topic}. "
        f"Include key facts, recent developments, and why it matters."
    )
    return {"background_info": response.content}


# 노드 2: 초고 작성
def write_draft(state: PipelineState):
    response = llm.invoke(
        f"Based on this background information:\n\n{state['background_info']}\n\n"
        f"Write a blog post about '{state['topic']}'. "
        f"Include an introduction, 2-3 main sections, and a conclusion. "
        f"Keep it around 500 words."
    )
    return {"draft": response.content}


# 그래프 조립
graph_builder = StateGraph(PipelineState)

graph_builder.add_node("get_topic_info", get_topic_info)
graph_builder.add_node("write_draft", write_draft)

graph_builder.add_edge(START, "get_topic_info")
graph_builder.add_edge("get_topic_info", "write_draft")
graph_builder.add_edge("write_draft", END)

graph = graph_builder.compile()

# 실행
result = graph.invoke({"topic": "LangGraph and AI Agent Orchestration"})

print("=== Background Info (first 300 chars) ===")
print(result["background_info"][:300])
print("\n=== Draft (first 500 chars) ===")
print(result["draft"][:500])

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 15.1

1. `PipelineState`에 `word_count: int` 필드를 추가하고, `write_draft` 노드에서 단어 수를 계산하여 저장하라.
2. `get_topic_info` 앞에 `validate_topic` 노드를 추가하여, 주제가 빈 문자열이면 기본값을 설정하라.
3. 다양한 주제로 파이프라인을 실행하여 결과를 비교하라.

In [ ]:
# Try it yourself


---
## 15.2 병렬 작성 노드 — Send API + Map-Reduce

블로그 글을 N개 섹션으로 분할하고 **병렬로** 작성한 후 합치는 패턴:

```
get_topic_info → dispatch_writers ──→ write_section (x N) → combine_sections
                                  ├─→ write_section
                                  └─→ write_section
```

핵심 개념:
- **`Send("node_name", data)`** — 동적으로 병렬 노드 발송 (Map)
- **`Annotated[list[str], operator.add]`** — 병렬 결과 자동 병합 (Reduce)
- **`with_structured_output()`** — LLM이 구조화된 데이터로 섹션 계획 생성

In [ ]:
import os
import operator
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# --- Structured Output: 섹션 계획 ---
class SectionPlan(BaseModel):
    """A plan for one section of the blog post."""
    title: str = Field(description="Section title")
    key_points: list[str] = Field(description="Key points to cover in this section")


class BlogOutline(BaseModel):
    """Complete outline for a blog post."""
    sections: list[SectionPlan] = Field(description="List of sections to write")


# --- 상태 정의 ---
class PipelineState(TypedDict):
    topic: str
    background_info: str
    outline: list[dict]            # 섹션 계획 목록
    sections: Annotated[list[str], operator.add]  # 병렬 결과 자동 병합!
    combined_draft: str


# write_section 노드의 입력 상태 (Send로 전달)
class SectionState(TypedDict):
    topic: str
    background_info: str
    section_title: str
    section_key_points: list[str]
    sections: Annotated[list[str], operator.add]


# --- 노드 정의 ---
def get_topic_info(state: PipelineState):
    topic = state["topic"]
    response = llm.invoke(
        f"Provide a concise background summary (3-4 paragraphs) about: {topic}. "
        f"Include key facts, recent developments, and why it matters."
    )
    return {"background_info": response.content}


def dispatch_writers(state: PipelineState):
    """LLM이 구조화된 섹션 계획을 생성하고, Send로 병렬 노드 발송"""
    planner = llm.with_structured_output(BlogOutline)
    outline = planner.invoke(
        f"Based on this background:\n{state['background_info']}\n\n"
        f"Create a blog post outline about '{state['topic']}' with exactly 3 sections. "
        f"Each section should have a title and 2-3 key points."
    )

    # Send API — 각 섹션을 병렬 노드로 전송!
    return [
        Send("write_section", {
            "topic": state["topic"],
            "background_info": state["background_info"],
            "section_title": section.title,
            "section_key_points": section.key_points,
            "sections": [],  # 초기값 (reducer가 병합)
        })
        for section in outline.sections
    ]


def write_section(state: SectionState):
    """병렬 실행되는 섹션 작성 노드 (Map)"""
    key_points = "\n".join(f"- {kp}" for kp in state["section_key_points"])
    response = llm.invoke(
        f"Write a blog post section about '{state['topic']}'.\n\n"
        f"Section title: {state['section_title']}\n"
        f"Key points to cover:\n{key_points}\n\n"
        f"Background context:\n{state['background_info'][:500]}\n\n"
        f"Write 150-200 words. Start with the section heading as '## {state['section_title']}'."
    )
    # Annotated[list[str], operator.add] 덕분에 자동 병합!
    return {"sections": [response.content]}


def combine_sections(state: PipelineState):
    """병렬 결과를 하나의 초고로 합침 (Reduce)"""
    combined = f"# {state['topic']}\n\n"
    combined += "\n\n".join(state["sections"])
    return {"combined_draft": combined}


# --- 그래프 조립 ---
graph_builder = StateGraph(PipelineState)

graph_builder.add_node("get_topic_info", get_topic_info)
graph_builder.add_node("dispatch_writers", dispatch_writers)
graph_builder.add_node("write_section", write_section)
graph_builder.add_node("combine_sections", combine_sections)

graph_builder.add_edge(START, "get_topic_info")
graph_builder.add_edge("get_topic_info", "dispatch_writers")
# dispatch_writers → write_section: Send API가 자동으로 라우팅 (엣지 불필요)
graph_builder.add_edge("write_section", "combine_sections")
graph_builder.add_edge("combine_sections", END)

graph = graph_builder.compile()

# 실행
result = graph.invoke({"topic": "Microservices Architecture Best Practices"})

print(f"\u2705 {len(result['sections'])} sections written in parallel!")
print(f"\nCombined draft length: {len(result['combined_draft'])} chars")
print("\n" + result["combined_draft"][:800])

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 15.2

1. `BlogOutline`의 섹션 수를 5개로 늘려보고, 병렬 실행 결과를 확인하라.
2. `write_section` 노드에 `section_index: int`를 추가하여 순서를 보장하라.
3. `operator.add` 대신 카스텀 리듀서를 작성하여 중복 제거 로직을 추가해 보라.

In [ ]:
# Try it yourself


---
## 15.3 Human-in-the-loop — interrupt + Command resume

초고 작성 후 **사람의 피드백**을 받고 최종본을 생성하는 패턴:

```
write_draft → human_feedback(interrupt!) → [사람 피드백] → finalize_post → END
```

핵심 개념:
- **`interrupt(value)`** — 그래프 실행 일시 중단, 값을 사용자에게 반환
- **`Command(resume=feedback)`** — 피드백을 전달하며 실행 재개
- **`SqliteSaver`** — 인터럽트 사이 상태 보존 (필수!)

In [ ]:
import os
import sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


class ReviewState(TypedDict):
    topic: str
    draft: str
    feedback: str
    final_post: str


def write_draft(state: ReviewState):
    response = llm.invoke(
        f"Write a short blog post (300 words) about: {state['topic']}"
    )
    return {"draft": response.content}


def human_feedback(state: ReviewState):
    """interrupt()로 사람에게 초고를 보여주고 피드백 대기"""
    feedback = interrupt(
        f"\n===== DRAFT FOR REVIEW =====\n"
        f"{state['draft'][:500]}...\n"
        f"============================\n"
        f"Please provide your feedback:"
    )
    return {"feedback": feedback}


def finalize_post(state: ReviewState):
    """\ud53c\ub4dc\ubc31\uc744 \ubc18\uc601\ud558\uc5ec \ucd5c\uc885\ubcf8 \uc0dd\uc131"""
    response = llm.invoke(
        f"Here is a blog post draft:\n\n{state['draft']}\n\n"
        f"The reviewer provided this feedback:\n{state['feedback']}\n\n"
        f"Please revise the post according to the feedback and return the final version."
    )
    return {"final_post": response.content}


# 그래프 조립
graph_builder = StateGraph(ReviewState)

graph_builder.add_node("write_draft", write_draft)
graph_builder.add_node("human_feedback", human_feedback)
graph_builder.add_node("finalize_post", finalize_post)

graph_builder.add_edge(START, "write_draft")
graph_builder.add_edge("write_draft", "human_feedback")
graph_builder.add_edge("human_feedback", "finalize_post")
graph_builder.add_edge("finalize_post", END)

# 체크포인터 필수! (interrupt 사이 상태 보존)
conn = sqlite3.connect("pipeline_review.db", check_same_thread=False)
graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [ ]:
# 1단계: 초고 작성 → interrupt에서 멈춤
config = {"configurable": {"thread_id": "review_1"}}

result = graph.invoke(
    {"topic": "Why Python is great for beginners"},
    config=config,
)

# 상태 확인 — human_feedback에서 대기중
snapshot = graph.get_state(config)
print("Paused at:", snapshot.next)
print("\nDraft preview:")
print(snapshot.values["draft"][:300])

In [ ]:
# 2단계: 피드백 전달 → 최종본 생성
result = graph.invoke(
    Command(resume="Make it more concise. Add a code example. Use friendlier tone."),
    config=config,
)

# 완료 확인
snapshot = graph.get_state(config)
print("Status:", "COMPLETE" if not snapshot.next else f"Waiting at {snapshot.next}")
print("\n=== Final Post ===")
print(result["final_post"][:600])

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 15.3

1. 피드백에 `"reject"` 문자열이 포함되면 `write_draft`로 돌아가는 조건부 엣지를 추가해 보라.
2. `SqliteSaver` 대신 `InMemorySaver`를 사용하면 어떤 점이 달라지는가?
   - `from langgraph.checkpoint.memory import InMemorySaver`
3. `Command(resume={"approved": True, "score": 8})` 처럼 구조화된 피드백을 전달해 보라.

In [ ]:
# Try it yourself


---
## 15.4 완성된 파이프라인 — 모든 패턴 통합

앞서 배운 모든 패턴을 하나의 파이프라인으로 통합:

```
[START]
   │
[get_topic_info] ───── LLM 배경 조사
   │
[dispatch_writers] ──── with_structured_output + Send API
   │
[write_section] x N ─── 병렬 작성 (Map)
   │
[combine_sections] ──── 초고 병합 (Reduce)
   │
[human_feedback] ───── interrupt() 리뷰
   │
[finalize_post] ────── 피드백 반영 최종본
   │
[END]
```

In [ ]:
import os
import operator
import sqlite3
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Send, interrupt, Command
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"openai:{os.getenv('OPENAI_MODEL_NAME')}")


# === Structured Output 스키마 ===
class SectionPlan(BaseModel):
    title: str = Field(description="Section title")
    key_points: list[str] = Field(description="Key points to cover")


class BlogOutline(BaseModel):
    sections: list[SectionPlan] = Field(description="List of sections")


# === 상태 정의 ===
class FullPipelineState(TypedDict):
    topic: str
    background_info: str
    outline: list[dict]
    sections: Annotated[list[str], operator.add]  # Map-Reduce 리듀서
    combined_draft: str
    feedback: str
    final_post: str


class SectionWriteState(TypedDict):
    topic: str
    background_info: str
    section_title: str
    section_key_points: list[str]
    sections: Annotated[list[str], operator.add]


# === 노드 정의 ===
def get_topic_info(state: FullPipelineState):
    """1단계: 주제 배경 조사"""
    response = llm.invoke(
        f"Provide a concise background summary (3-4 paragraphs) about: {state['topic']}. "
        f"Include key facts, recent developments, and why it matters."
    )
    return {"background_info": response.content}


def dispatch_writers(state: FullPipelineState):
    """2단계: 구조화된 아웃라인 생성 + Send API로 병렬 발송"""
    planner = llm.with_structured_output(BlogOutline)
    outline = planner.invoke(
        f"Based on this background:\n{state['background_info']}\n\n"
        f"Create a blog post outline about '{state['topic']}' with exactly 3 sections. "
        f"Each section should have a title and 2-3 key points."
    )
    print(f"\ud83d\udcdd Dispatching {len(outline.sections)} parallel writers...")
    for s in outline.sections:
        print(f"   - {s.title}")

    return [
        Send("write_section", {
            "topic": state["topic"],
            "background_info": state["background_info"],
            "section_title": section.title,
            "section_key_points": section.key_points,
            "sections": [],
        })
        for section in outline.sections
    ]


def write_section(state: SectionWriteState):
    """3단계: 병렬 섹션 작성 (Map)"""
    key_points = "\n".join(f"- {kp}" for kp in state["section_key_points"])
    response = llm.invoke(
        f"Write a blog post section about '{state['topic']}'.\n\n"
        f"Section title: {state['section_title']}\n"
        f"Key points:\n{key_points}\n\n"
        f"Background:\n{state['background_info'][:500]}\n\n"
        f"Write 150-200 words. Start with '## {state['section_title']}'."
    )
    return {"sections": [response.content]}


def combine_sections(state: FullPipelineState):
    """4단계: 병렬 결과 합침 (Reduce)"""
    combined = f"# {state['topic']}\n\n"
    combined += "\n\n".join(state["sections"])
    print(f"\u2705 Combined {len(state['sections'])} sections into draft")
    return {"combined_draft": combined}


def human_feedback(state: FullPipelineState):
    """5단계: 사람 리뷰 (interrupt)"""
    draft_preview = state["combined_draft"][:1000]
    feedback = interrupt(
        f"\n{'='*50}\n"
        f"DRAFT FOR REVIEW\n"
        f"{'='*50}\n"
        f"{draft_preview}...\n"
        f"{'='*50}\n"
        f"Please provide your feedback:"
    )
    return {"feedback": feedback}


def finalize_post(state: FullPipelineState):
    """6단계: 피드백 반영 최종본 생성"""
    response = llm.invoke(
        f"Here is a blog post draft:\n\n{state['combined_draft']}\n\n"
        f"Reviewer feedback:\n{state['feedback']}\n\n"
        f"Revise the post according to the feedback. "
        f"Return the complete final version with all sections."
    )
    return {"final_post": response.content}


# === 그래프 조립 ===
graph_builder = StateGraph(FullPipelineState)

graph_builder.add_node("get_topic_info", get_topic_info)
graph_builder.add_node("dispatch_writers", dispatch_writers)
graph_builder.add_node("write_section", write_section)
graph_builder.add_node("combine_sections", combine_sections)
graph_builder.add_node("human_feedback", human_feedback)
graph_builder.add_node("finalize_post", finalize_post)

graph_builder.add_edge(START, "get_topic_info")
graph_builder.add_edge("get_topic_info", "dispatch_writers")
# dispatch_writers → write_section: Send API 자동 라우팅
graph_builder.add_edge("write_section", "combine_sections")
graph_builder.add_edge("combine_sections", "human_feedback")
graph_builder.add_edge("human_feedback", "finalize_post")
graph_builder.add_edge("finalize_post", END)

# 체크포인터 (HITL 필수)
conn = sqlite3.connect("pipeline_full.db", check_same_thread=False)
graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [ ]:
# 1단계: 파이프라인 실행 → human_feedback에서 멈춤
config = {"configurable": {"thread_id": "full_pipeline_1"}}

result = graph.invoke(
    {"topic": "Building AI Agents with LangGraph"},
    config=config,
)

snapshot = graph.get_state(config)
print(f"Paused at: {snapshot.next}")
print(f"Sections written: {len(snapshot.values.get('sections', []))}")
print(f"Draft length: {len(snapshot.values.get('combined_draft', ''))} chars")
print("\n--- Draft Preview ---")
print(snapshot.values["combined_draft"][:600])

In [ ]:
# 2단계: 피드백 전달 → 최종본 생성
result = graph.invoke(
    Command(resume="Add a practical code example in each section. Make the tone more engaging."),
    config=config,
)

snapshot = graph.get_state(config)
print(f"Status: {'COMPLETE' if not snapshot.next else f'Waiting at {snapshot.next}'}")
print(f"\n{'='*50}")
print("FINAL POST")
print(f"{'='*50}")
print(result["final_post"][:1000])

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Exercise 15.4

1. `human_feedback` 노드에서 조건부 엣지를 추가하여 `"approve"`면 바로 END로, 아니면 `finalize_post`로 가도록 하라.
2. 파이프라인에 `translate_post` 노드를 추가하여 최종본을 다른 언어로 번역하라.
3. `get_state_history()`를 사용하여 파이프라인의 각 단계별 상태 변화를 추적하라.

In [ ]:
# Try it yourself


---
## 15.5 프로덕션 배포 구조 — langgraph.json + graph.py

실제 프로덕션에서는 노트북이 아닌 **파일 분리** 구조로 배포합니다.

### 디렉토리 구조

```
my_pipeline/
├── langgraph.json       # 파이프라인 진입점 설정
├── graph.py             # 그래프 정의 (nodes, edges, compile)
├── state.py             # 상태 스키마 (TypedDict / Pydantic)
├── nodes.py             # 노드 함수들
├── prompts.py           # LLM 프롬프트 템플릿
└── requirements.txt     # 의존성
```

### langgraph.json

```json
{
    "dependencies": ["."],
    "graphs": {
        "blog_pipeline": "./graph.py:graph"
    },
    "env": ".env"
}
```

`"blog_pipeline": "./graph.py:graph"` — `graph.py` 파일의 `graph` 변수가 진입점

### graph.py 예시

```python
# graph.py
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from state import FullPipelineState, SectionWriteState
from nodes import (
    get_topic_info,
    dispatch_writers,
    write_section,
    combine_sections,
    human_feedback,
    finalize_post,
)

graph_builder = StateGraph(FullPipelineState)

graph_builder.add_node("get_topic_info", get_topic_info)
graph_builder.add_node("dispatch_writers", dispatch_writers)
graph_builder.add_node("write_section", write_section)
graph_builder.add_node("combine_sections", combine_sections)
graph_builder.add_node("human_feedback", human_feedback)
graph_builder.add_node("finalize_post", finalize_post)

graph_builder.add_edge(START, "get_topic_info")
graph_builder.add_edge("get_topic_info", "dispatch_writers")
graph_builder.add_edge("write_section", "combine_sections")
graph_builder.add_edge("combine_sections", "human_feedback")
graph_builder.add_edge("human_feedback", "finalize_post")
graph_builder.add_edge("finalize_post", END)

# compile — 체크포인터는 배포 환경에서 설정
graph = graph_builder.compile()
```

### LangGraph CLI로 로컬 테스트

```bash
# 로컬 개발 서버 시작
langgraph dev

# Studio UI에서 시각적 테스트
# http://localhost:8123 접속
```

### 배포 옵션

| 방식 | 설명 |
|------|------|
| `langgraph dev` | 로컬 개발 + Studio UI |
| `langgraph up` | Docker 컨테이너 실행 |
| LangGraph Cloud | 관리형 클라우드 배포 |
| Self-hosted | 커스텀 인프라에 배포 |

### 핵심 포인트

- **노트북 = 프로토타이핑**, **graph.py = 프로덕션**
- `langgraph.json`이 진입점을 정의 — 여러 그래프 등록 가능
- 체크포인터, 스토어는 배포 환경에서 설정 (Postgres 권장)
- `InMemorySaver` → 개발용, `SqliteSaver` → 로컬, `PostgresSaver` → 프로덕션

### Exercise 15.5

1. 위의 파이프라인을 `graph.py`, `state.py`, `nodes.py`로 분리하여 디렉토리 구조를 만들어 보라.
2. `langgraph.json`에 두 번째 그래프(`summary_pipeline`)를 등록해 보라.
3. `langgraph dev` 명령으로 Studio UI를 실행하고 그래프를 시각적으로 확인해 보라.

In [ ]:
# Try it yourself


---
## Final Exercises — 종합 실습

### 과제 1: 뉴스레터 파이프라인 (★★☆)

`Blog Post Generator`를 변형하여 **뉴스레터 생성기**를 만들어라:
1. `get_topic_info` → 주간 뉴스 요약
2. `dispatch_writers` → 카테고리별 분배 (Tech, Business, Science)
3. `write_section` → 각 카테고리 병렬 작성
4. `combine_sections` → 뉴스레터 편집
5. `human_feedback` → 에디터 리뷰

In [ ]:
# 과제 1: 여기에 작성


### 과제 2: 에러 처리 + 재시도 로직 (★★☆)

파이프라인에 **에러 처리**를 추가하라:
1. `write_section`에서 LLM 호출 실패 시 `try/except`로 기본값 반환
2. `combine_sections`에서 빈 섹션 감지 → 재작성 요청
3. 조건부 엣지로 최대 2회 재시도 후 처리를 계속하는 로직 구현

In [ ]:
# 과제 2: 여기에 작성


### 과제 3: 다중 리뷰어 파이프라인 (★★★)

`human_feedback` 노드를 확장하여 **다중 리뷰 사이클**을 구현하라:
1. 첫 번째 `interrupt`: 기술 리뷰어가 정확성 검토
2. 두 번째 `interrupt`: 에디터가 문체/톤 검토
3. 각 리뷰어의 피드백을 모두 반영하여 최종본 생성
4. `Command(resume={"reviewer": "tech", "feedback": "..."})` 구조화된 피드백 사용

In [ ]:
# 과제 3: 여기에 작성


### 과제 4: 프로덕션 배포 설계 (★★★)

15.4의 완성된 파이프라인을 프로덕션 구조로 재구성하라:
1. `state.py` — `FullPipelineState`, `SectionWriteState` 정의
2. `nodes.py` — 모든 노드 함수
3. `graph.py` — 그래프 조립 + `graph = graph_builder.compile()`
4. `langgraph.json` — 진입점 설정
5. `langgraph dev`로 로컬 실행 테스트

In [ ]:
# 과제 4: 여기에 작성
